# 00. Prepare simulation regions and reference genotypes

Run this notebook once on the MSK cluster before the simulation jobs. It samples 500 regions and extracts their variants from each reference panel.

1. **Regions:** Sample 500 LD blocks (`set.seed(123)`) from `/home/sunh4/sunh4/RSS_QC/ld_meta_file.tsv`.
2. **Variants:** Sample 4,000 ADSP variants per block (`slice_sample(n = 4000)`) from `/home/sunh4/snp_info/chrom_<chr>_snp.tsv` → `trim_genotype/snp_info_random/snp_<i>.{txt,tsv}`.
3. **Reference panels:** Extract the sampled SNP IDs from UK Biobank and 1000 Genomes EUR with PLINK `--extract`. Extract the corresponding position ranges from GTEx v8 with `plink2 --extract range`, because those files lack rsIDs.
4. **UKB:** Lift the coordinates from hg19 to hg38 and select 342,053 unrelated participants of European ancestry.

Each simulation job applies its MAF and missingness filters and intersects the panels. The final set has fewer than 4,000 variants per region.

The code cells come from these original MSK notebooks and use their cluster paths.
- `simulation_final/1.ADSP_RSS_QC_Simulation_Random.ipynb` (steps 1–3, UKB / 1000G)
- `simulation_final/9.Stocsum_UKBB_Gtex.ipynb` (step 3, GTEx)
- `2026_Simulation/1.Stoc_LD_validation.ipynb` (step 4)

**Outputs used downstream** (copied to Minerva under `/sc/arion/projects/ad-omics/haochen/rss_qc/genotypes/`):

| MSK output | Minerva copy read by the Sinai notebooks |
|---|---|
| `trim_genotype/ukbb_unrelated_trimed/result/UKBB_<i>` | `genotypes/ukbb_unrelated_trimed/result/UKBB_<i>` |
| `trim_genotype/kg_trimed_random/EUR_KG_<i>` | `genotypes/kg_trimed_random/EUR_KG_<i>` |
| `trim_genotype/gtex_trimed_random/gtex_<i>` | `genotypes/gtex_trimed_random/gtex_<i>` |

Later notebooks create `gtex_EUR_trimed_random` (GTEx EUR only, `07` / `08`) and `ukbb_unrelated_trimed_sample_1k` (1,000 UKB individuals, `13`). The historical `01` notebook downloaded the ADSP sketch. The MSK notebook trimmed it to the same 500 regions to produce `genotypes/ADSP_sketch_trimed/result/ADSP_<i>`. See `2026_Simulation/5.B.Multi_Mix_Data.ipynb` (section "Keep the same variants").

## Input genotype files

UK Biobank: `/mnt/vast/hpc/csg/xc2270/ukb_impute/LDBlock`

1000 Genomes: `/home/sunh4/1000G_genome_b38/genotype_block/chr1_genotype_by_region`
    
/mnt/vast/hpc/csg/data_public/1000G_EUR_Phase3_plink

## 1–2. Pick 500 regions and sample 4,000 ADSP variants per region
*Source: `1.ADSP_RSS_QC_Simulation_Random.ipynb`, cells 6–7.* Writes `snp_info_random/snp_<i>.txt` (SNP IDs for plink) and `snp_<i>.tsv` (full table).

In [23]:
library(tidyverse)
set.seed(123)

blocks = read_tsv("/home/sunh4/sunh4/RSS_QC/ld_meta_file.tsv")

sample_id = sample(1:nrow(blocks), 500)

blocks = blocks[sample_id,]

for(i in 1:500){

    block_chrom = str_remove(blocks$`#chrom`[i], "chr")
    
    block_start = blocks$start[i]
    
    block_end = blocks$end[i]
    
    adsp_path = paste0("/home/sunh4/snp_info/chrom_", block_chrom, "_snp.tsv")
    
    adsp_var = read_tsv(adsp_path)
    
    adsp_var = adsp_var %>% filter(BP >= block_start & BP <= block_end) %>% arrange(BP)  %>%
      slice_sample(n = 4000) %>% mutate(variant = paste0(CHR, ":", BP, ":", A2, ":", A1))
    
    snp_path = paste0("/home/sunh4/rss_qc/output/trim_genotype/snp_info_random/snp_", i, ".txt")
    
    snp_tb = paste0("/home/sunh4/rss_qc/output/trim_genotype/snp_info_random/snp_", i, ".tsv")
    
    adsp_var$SNP %>% writeLines(snp_path)
    
    adsp_var %>% write_tsv(snp_tb)

}

Rows: 1488206 Columns: 5
── Column specification ────────────────────────────────────────────────────────
Delimiter: "\t"
chr (3): SNP, A1, A2
dbl (2): CHR, BP

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 1324114 Columns: 5
── Column specification ────────────────────────────────────────────────────────
Delimiter: "\t"
chr (3): SNP, A1, A2
dbl (2): CHR, BP

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 594073 Columns: 5
── Column specification ────────────────────────────────────────────────────────
Delimiter: "\t"
chr (3): SNP, A1, A2
dbl (2): CHR, BP

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 1215335 Columns: 5
── Column specification ───────

Record the same 500 blocks by reusing the seed. *Cells 8–9.*

In [7]:
library(tidyverse)
set.seed(123)

blocks = read_tsv("/home/sunh4/sunh4/RSS_QC/ld_meta_file.tsv")

sample_id = sample(1:nrow(blocks), 500)

blocks = blocks[sample_id,]

Rows: 1361 Columns: 4
── Column specification ────────────────────────────────────────────────────────
Delimiter: "\t"
chr (2): #chrom, path
dbl (2): start, end

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


In [8]:
sample_id

[1]  415  463  179  526  195  938 1142 1323 1253 1268 1038  665  602  709 1011
 [16] 1115  953  348 1017  840   26  519  211  932  593  555  373  844 1167  544
 [31]  490  905  937 1047  923  956  309 1159 1248  166  217 1314  581   72  588
 [46]  141  722  859  153  294  277   41  431   90  316 1247  528 1140  774  747
 [61]  456  598 1063 1183  752 1233  374   34  516   13   69  755  409 1113  928
 [76] 1006  537  983  291  671  121 1134 1182 1088  480   67 1014 1109  165 1075
 [91] 1098 1202  236  610  330  726  127  212  686  814  931  878  243  847 1137
[106] 1284  619  477  151  666  767  160  155  426 1029  326  789  985   39  822
[121]  986  137  455  589 1107 1243  196  680  500 1244  344  459  944 1044  872
[136] 1357 1188 1076  876  534  177  554  827   84  633  597  706 1255  710  712
[151]  672 1021  398 1013 1305  381   40  522  473  200 1149  265  775  186  458
[166] 1176  831 1078  919  235  185  413  627 1204 1194  205  779  564  794 1289
[181] 1250  468  920 1081  457  617  357  279  270 1260  646  347 1153  218  618
[196]  698  337  797 1341  539  981 1326  724 1282  861  657  960 1178  660  163
[211]  238  578 1267  117  648 1079 1216  557 1361  957  873  688  757  988  447
[226] 1128  821 1195  711  349 1221  386   24  945 1023 1273  170 1213  422  508
[241]   64   80  548  987  475 1283  765  479  791 1330  807  292  297  860  605
[256]  637 1299 1126 1214 1057   83  866 1311 1127   76 1054  946 1295  323  115
[271]  850  608  682 1356  397  989  392 1337  744 1259  106   11  625 1130  403
[286]  461 1316   31   94   16  178 1131  524  924  204 1335 1171  384  315  259
[301]  494 1016   10 1125  402  108    8  626  261  541 1246  282 1067  696  667
[316]  990  452  856  622 1065  891 1034 1350  793 1360  278  241 1129  679   37
[331] 1263  566   19  378  549   48  464  393  139  299  371  670  189  970  311
[346] 1019  418  569  382   38 1218  319 1031  846  838 1102  642  120 1212  331
[361]  533  441  199  499  599 1318 1105  714  677   81   55  134  424  756    6
[376]  128  879  668  800   49  739  476  239  340  193 1348 1230  303  148  898
[391]  190  624  191  446  119 1189 1225  966   59  817  903   61 1123 1056 1110
[406] 1066  535 1092  930  600  644  950 1190 1166  763  203  758  246  440  690
[421]  251  560  643  545  162  322  576  168  442  788   78 1039  493  999 1086
[436]  445   95  918 1025  379  342  221 1048  161  620  448  242  693 1262  536
[451]  828  407  229  224  785  474  699 1000  171   23 1168  484  301 1147   79
[466]  511  507  164  237  579 1111  959  929  730  796  209  997  914  358  650
[481]  877  883 1310  129  848  450  232  334  396  888 1342  639 1232  264  697
[496]  201   52  225 1276 1234

Record each region's chromosome in the order used by the PLINK loops below. *Cells 10–11.*

In [4]:
library(tidyverse)
set.seed(123)

blocks = read_tsv("/home/sunh4/sunh4/RSS_QC/ld_meta_file.tsv")

sample_id = sample(1:nrow(blocks), 500)

blocks = blocks[sample_id,]
blocks %>% mutate(chrom = str_remove(`#chrom`, "chr")) %>% pull(chrom) %>% writeLines("/home/sunh4/rss_qc/output/trim_genotype/snp_info_random/chrom_region.txt")

Rows: 1361 Columns: 4
── Column specification ────────────────────────────────────────────────────────
Delimiter: "\t"
chr (2): #chrom, path
dbl (2): start, end

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


## 3a. Trim UK Biobank (hg19 imputed) to the sampled variants
*Source: `1.ADSP_RSS_QC_Simulation_Random.ipynb`, cell 13.* → `trim_genotype/ukbb_trimed_random/UKBB_<i>`

In [ ]:
#!/bin/bash -l
# NOTE the -l flag!
#SBATCH -t 128:00:00
#SBATCH --mem=30000
#SBATCH -J trim
#SBATCH -o trim.%j.out
#SBATCH -e trim.%j.err
module load Plink
export PATH=$HOME/miniforge3/envs/space/bin:$PATH


mapfile -t chroms < /home/sunh4/rss_qc/output/trim_genotype/snp_info_random/chrom_region.txt   # zero-based bash array


for i in {1..500}; do

    chrom="${chroms[$((i-1))]}"           # line i → index i-1

    bfile="/mnt/vast/hpc/csg/data_public/UKBiobank/20250519_imputed/ukb_impute/chr${chrom}_imp_info08_maf001"

    plink --bfile "${bfile}" \
          --extract /home/sunh4/rss_qc/output/trim_genotype/snp_info_random/snp_${i}.txt \
          --make-bed \
          --out /home/sunh4/rss_qc/output/trim_genotype/ukbb_trimed_random/UKBB_${i}
cd 
done

## 3b. Trim 1000 Genomes EUR (hg38) to the sampled variants
*Source: `1.ADSP_RSS_QC_Simulation_Random.ipynb`, cell 15.* → `trim_genotype/kg_trimed_random/EUR_KG_<i>`

In [ ]:
#!/bin/bash -l
# NOTE the -l flag!
#SBATCH -t 128:00:00
#SBATCH --mem=30000
#SBATCH -J 1KG_trim
#SBATCH -o trim.%j.out
#SBATCH -e trim.%j.err
module load Plink
export PATH=$HOME/miniforge3/envs/space/bin:$PATH


mapfile -t chroms < /home/sunh4/rss_qc/output/trim_genotype/snp_info_random/chrom_region.txt   # zero-based bash array


for i in {1..500}; do

    chrom="${chroms[$((i-1))]}"           # line i → index i-1

    bfile="/home/sunh4/rss_qc/output/1000G_EUR_hg38/1000G_b38_EUR_chr${chrom}"

    plink --bfile "${bfile}" \
          --extract /home/sunh4/rss_qc/output/trim_genotype/snp_info_random/snp_${i}.txt \
          --make-bed \
          --out /home/sunh4/rss_qc/output/trim_genotype/kg_trimed_random/EUR_KG_${i}

done

## 3c. Trim GTEx v8 by position range
*Source: `9.Stocsum_UKBB_Gtex.ipynb`, cells 5–8.* GTEx has no rsIDs, so each region's min–max position is written to `snp_<i>_range.txt` and used with `plink2 --extract range` → `trim_genotype/gtex_trimed_random/gtex_<i>`. The second cell writes `base_script` in `Gtex_partition/code`.

In [6]:
library(tidyverse)
files =  list.files("/home/sunh4/rss_qc/output/trim_genotype/snp_info_random", pattern = ".tsv", full.names = T)
for (file in files) {
  
  snp_tb <- read_tsv(file, show_col_types = FALSE)
  
  position_tb <- snp_tb %>%
    summarise(
      CHR   = as.character(first(CHR)),
      START = min(BP, na.rm = TRUE),
      END   = max(BP, na.rm = TRUE)
    )
  
  save_path <- str_replace(file, "\\.tsv$", "_range.txt")
  
  write_delim(
    position_tb,
    save_path,
    delim = " ",
    col_names = FALSE
  )
}

In [ ]:
work_dir="/home/sunh4/rss_qc/output/gtex/Gtex_partition/"
job="partitioning"
mkdir -p ${work_dir}
mkdir -p ${work_dir}/code
mkdir -p ${work_dir}/log
mkdir -p ${work_dir}/result
cd ${work_dir}/code
cat << 'EOF' > base_script
#!/bin/bash -l
# NOTE the -l flag!
#SBATCH -t 128:00:00
#SBATCH --mem=30000
#SBATCH -J trim
#SBATCH -o trim.%j.out
#SBATCH -e trim.%j.err
export PATH=$HOME/miniforge3/envs/space/bin:$PATH


mapfile -t chroms < /home/sunh4/rss_qc/output/trim_genotype/snp_info_random/chrom_region.txt   # zero-based bash array


for i in {1..500}; do

    chrom="${chroms[$((i-1))]}"           # line i → index i-1

    bfile="/data1/deyk/GTEx/genotype/GTEx_v8_genotype.${chrom}"

    plink2 --bfile "${bfile}" \
          --extract range /home/sunh4/rss_qc/output/trim_genotype/snp_info_random/snp_${i}_range.txt \
          --make-bed \
          --out /home/sunh4/rss_qc/output/trim_genotype/gtex_trimed_random/gtex_${i}

done
EOF

## 4a. Lift the trimmed UKB genotypes hg19 → hg38
*Source: `2026_Simulation/1.Stoc_LD_validation.ipynb`, cells 7–10.* → `trim_genotype/ukbb_trimed_random_lifted/result/UKBB_<i>` (unmapped variants dropped; a `summary.txt` per region).

In [ ]:
src_dir="/home/sunh4/rss_qc/output/trim_genotype/ukbb_trimed_random"
out_dir="/home/sunh4/rss_qc/output/trim_genotype/ukbb_trimed_random_lifted"
tmp_dir="${out_dir}/tmp"
chain="${out_dir}/hg19ToHg38.over.chain.gz"
liftover_bin="${out_dir}/liftOver"

mkdir -p "${out_dir}" "${tmp_dir}"

# 1) Put UCSC binaries/chain here once:
#    liftOver executable
#    hg19ToHg38.over.chain.gz
#
# Example download commands (run once if needed):
# wget -O "${liftover_bin}" https://hgdownload.soe.ucsc.edu/admin/exe/linux.x86_64.v479/liftOver
# chmod +x "${liftover_bin}"
# wget -O "${chain}" https://hgdownload.soe.ucsc.edu/goldenPath/hg19/liftOver/hg19ToHg38.over.chain.gz

In [ ]:
work_dir="/home/sunh4/rss_qc/output/trim_genotype/ukbb_trimed_random_lifted"
mkdir -p ${work_dir}/code
mkdir -p ${work_dir}/log
mkdir -p ${work_dir}/result
cd ${work_dir}/code

cat << 'EOF' > base_script
#!/bin/bash
#SBATCH --time=18:00:00
#SBATCH --mem=40G
#SBATCH --job-name=ukbb_lift
#SBATCH --cpus-per-task=1
#SBATCH --array=1-500
#SBATCH --output=WORK_DIR/log/ukbb_lift.%A_%a.out
#SBATCH --error=WORK_DIR/log/ukbb_lift.%A_%a.err

set -euo pipefail

export PATH=$HOME/miniforge3/envs/space/bin:$PATH

src_dir="/home/sunh4/rss_qc/output/trim_genotype/ukbb_trimed_random"
out_dir="WORK_DIR/result"
tmp_dir="WORK_DIR/result/tmp_${SLURM_ARRAY_TASK_ID}"

mkdir -p ${out_dir}
mkdir -p ${tmp_dir}

prefix="UKBB_${SLURM_ARRAY_TASK_ID}"
in_pref="${src_dir}/${prefix}"
out_pref="${out_dir}/${prefix}"

liftOver_bin="/home/sunh4/rss_qc/output/trim_genotype/ukbb_trimed_random_lifted/liftOver"
chain_file="/home/sunh4/rss_qc/output/trim_genotype/ukbb_trimed_random_lifted/hg19ToHg38.over.chain.gz"

if [ ! -f "${in_pref}.bim" ]; then
    echo "Missing ${in_pref}.bim"
    exit 1
fi

if [ ! -f "${in_pref}.bed" ]; then
    echo "Missing ${in_pref}.bed"
    exit 1
fi

if [ ! -f "${in_pref}.fam" ]; then
    echo "Missing ${in_pref}.fam"
    exit 1
fi

if [ ! -x "${liftOver_bin}" ]; then
    echo "liftOver not executable: ${liftOver_bin}"
    exit 1
fi

if [ ! -f "${chain_file}" ]; then
    echo "Missing chain file: ${chain_file}"
    exit 1
fi

echo "Processing ${prefix}"

# BIM -> BED for liftOver
awk 'BEGIN{OFS="\t"}
{
    chr=$1
    if (chr !~ /^chr/) chr="chr" chr
    start=$4-1
    end=$4
    id=$2
    print chr, start, end, id
}' "${in_pref}.bim" > "${tmp_dir}/${prefix}.hg19.bed"

# liftOver hg19 -> hg38
"${liftOver_bin}" \
    "${tmp_dir}/${prefix}.hg19.bed" \
    "${chain_file}" \
    "${tmp_dir}/${prefix}.hg38.bed" \
    "${tmp_dir}/${prefix}.unmapped.bed"

# mapped variant ids
cut -f4 "${tmp_dir}/${prefix}.hg38.bed" > "${tmp_dir}/${prefix}.mapped.ids"

# subset genotype to mapped variants only
plink \
    --bfile "${in_pref}" \
    --extract "${tmp_dir}/${prefix}.mapped.ids" \
    --make-bed \
    --out "${tmp_dir}/${prefix}.tmp"

# rebuild lifted BIM
awk 'BEGIN{OFS="\t"}
    NR==FNR {
        chr=$1
        id=$2
        cm=$3
        bp=$4
        a1=$5
        a2=$6
        bim_chr[id]=chr
        bim_cm[id]=cm
        bim_a1[id]=a1
        bim_a2[id]=a2
        next
    }
    {
        chr=$1
        sub(/^chr/, "", chr)
        newbp=$2+1
        id=$4
        print chr, id, bim_cm[id], newbp, bim_a1[id], bim_a2[id]
    }' "${in_pref}.bim" "${tmp_dir}/${prefix}.hg38.bed" > "${out_pref}.bim"

mv "${tmp_dir}/${prefix}.tmp.bed" "${out_pref}.bed"
mv "${tmp_dir}/${prefix}.tmp.fam" "${out_pref}.fam"

cp "${tmp_dir}/${prefix}.unmapped.bed" "${out_pref}.unmapped.bed"
cp "${tmp_dir}/${prefix}.hg38.bed" "${out_pref}.mapped.hg38.bed"

n_in=$(wc -l < "${in_pref}.bim")
n_out=$(wc -l < "${out_pref}.bim")
echo "${prefix}: mapped ${n_out} / ${n_in}" > "${out_pref}.summary.txt"

rm -f "${tmp_dir}/${prefix}.tmp.log" "${tmp_dir}/${prefix}.tmp.nosex"
rm -rf "${tmp_dir}"

echo "Done ${prefix}"
EOF

base_sh="base_script"
output_script="run_ukbb_liftover.sh"
cat ${base_sh} | sed "s|WORK_DIR|${work_dir}|g" > ${output_script}
sbatch ${output_script}

## 4b. Subset to unrelated UK Biobank participants of European ancestry (n = 342,053)
*Source: `2026_Simulation/1.Stoc_LD_validation.ipynb`, cells 12–13.* → `trim_genotype/ukbb_unrelated_trimed/result/UKBB_<i>`, the UK Biobank input for the simulations.

### IDs of unrelated participants

Sample ID file: `/data1/deyk/haochen/rss_qc/UKBB_407917_white_europeans_646469_variants_QCed_LD_prune.prune.in.estimate_kinship_kinship_cutoff_0.044.unrelated_id`

In [ ]:
work_dir="/data1/deyk/haochen/rss_qc/output/trim_genotype/ukbb_unrelated_trimed"
mkdir -p ${work_dir}/code
mkdir -p ${work_dir}/log
mkdir -p ${work_dir}/result
cd ${work_dir}/code

cat << 'EOF' > base_script
#!/bin/bash
#SBATCH --time=8:00:00
#SBATCH --mem=20G
#SBATCH --job-name=ukbb_keep_unrel
#SBATCH --cpus-per-task=1
#SBATCH --array=1-500
#SBATCH --output=WORK_DIR/log/ukbb_keep_unrel.%A_%a.out
#SBATCH --error=WORK_DIR/log/ukbb_keep_unrel.%A_%a.err

set -euo pipefail

export PATH=$HOME/miniforge3/envs/space/bin:$PATH

src_dir="/home/sunh4/rss_qc/output/trim_genotype/ukbb_trimed_random_lifted/result/"
out_dir="WORK_DIR/result"
keep_file="/data1/deyk/haochen/rss_qc/UKBB_407917_white_europeans_646469_variants_QCed_LD_prune.prune.in.estimate_kinship_kinship_cutoff_0.044.unrelated_id"

mkdir -p "${out_dir}"

prefix="UKBB_${SLURM_ARRAY_TASK_ID}"
in_pref="${src_dir}/${prefix}"
out_pref="${out_dir}/${prefix}"

if [ ! -f "${in_pref}.bed" ] || [ ! -f "${in_pref}.bim" ] || [ ! -f "${in_pref}.fam" ]; then
    echo "Missing input plink files for ${prefix}"
    exit 1
fi

if [ ! -f "${keep_file}" ]; then
    echo "Missing keep file: ${keep_file}"
    exit 1
fi

echo "Processing ${prefix}"

plink \
    --bfile "${in_pref}" \
    --keep "${keep_file}" \
    --make-bed \
    --out "${out_pref}"

n_before=$(wc -l < "${in_pref}.fam")
n_after=$(wc -l < "${out_pref}.fam")
echo "${prefix}: kept ${n_after} of ${n_before} samples" > "${out_pref}.summary.txt"

echo "Done ${prefix}"
EOF

base_sh="base_script"
output_script="run_keep_unrelated.sh"
cat ${base_sh} | sed "s|WORK_DIR|${work_dir}|g" > ${output_script}
sbatch ${output_script}